# Introduction and Project Setup

# Project 5 – Innovative Geospatial Programming  
## Wind Farm Site Suitability Analysis (Nebraska)

**Purpose:**  
To automate acquisition, preprocessing, and suitability analysis of key spatial datasets to identify high-potential wind farm locations.

**Key Layers:**  
- Wind Power Density (Global Wind Atlas)  
- Land Use Land Cover (Sentinel 10m LULC)  
- Transmission Lines (AGOL)  
- DEM (USGS 3DEP) + Slope  

**Software:**  
ArcGIS Pro, Python (ArcPy), ArcGIS Online

Setting up

In [1]:
# Import libraries
import arcpy
from arcgis.gis import GIS
import os
import requests
from arcpy.sa import ExtractByMask
from arcgis.gis import GIS
from arcpy.sa import Reclassify, RemapRange, Con, IsNull
from arcpy.sa import *
import arcgis.mapping

# Connect to AGOL using ArcGIS Pro credentials
gis = GIS("home")

# Confirm sign-in
gis.properties.portalName


'ArcGIS Online'

More setting up

In [2]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

print("GDB:", gdb)
print("Map:", m.name)

GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Map: Map


Import State boundary shapefile

In [3]:
# 1️⃣ EDIT THESE
item_id = "2399ad4266224fc5bf74e4b485dde52a"   # your AGOL polygon item id
out_name = "Nebraska"               # name inside your project GDB

# 2️⃣ Connect to the same portal that Pro is signed into
gis = GIS("home")

item = gis.content.get(item_id)
if item is None:
    raise ValueError(f"Could not find item with ID {item_id}. "
                     "Check portal sign-in and sharing.")

print(f"Found item: {item.title} ({item.type})")

# 3️⃣ Get the actual feature layer + URL
if hasattr(item, "layers") and len(item.layers) > 0:
    lyr = item.layers[0]   # first sublayer
    print(f"Using sublayer: {lyr.properties.name}")
    in_fc = lyr.url        # <-- use URL for ArcPy
else:
    # item itself might be a Feature Layer
    if hasattr(item, "url"):
        lyr = item
        in_fc = item.url
        print("Item is a Feature Layer; using its URL.")
    else:
        raise TypeError("Item does not have 'layers' or 'url'. "
                        "It may not be a feature layer/service.")

print(f"Feature layer URL for ArcPy: {in_fc}")

# 4️⃣ Figure out output path in the current project GDB
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase     # e.g. C:\...\YourProject.gdb

out_name_valid = arcpy.ValidateTableName(out_name, gdb)
out_fc = f"{gdb}\\{out_name_valid}"
print(f"Output feature class will be: {out_fc}")

# 5️⃣ Run FeatureClassToFeatureClass with error reporting
try:
    arcpy.conversion.FeatureClassToFeatureClass(
        in_features=in_fc,
        out_path=gdb,
        out_name=out_name_valid
    )
    print("FeatureClassToFeatureClass completed successfully.")
except Exception as e:
    print("❌ ArcPy raised an error.")
    print("Python exception:", e)
    print("\nArcPy messages:")
    print(arcpy.GetMessages(2))   # only error + warning messages
    raise

# 6️⃣ Add to active map (optional)
m = aprx.activeMap
if m:
    m.addDataFromPath(out_fc)
    print(f"Added '{out_name_valid}' to map: {m.name}")
else:
    print("No active map found; feature class created but not added to a map.")

Found item: Nebraska State Boundary (Feature Service)
Using sublayer: State Boundary DOT
Feature layer URL for ArcPy: https://gis.ne.gov/Enterprise/rest/services/Nebraska_State_Boundary/FeatureServer/0
Output feature class will be: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Nebraska
FeatureClassToFeatureClass completed successfully.
Added 'Nebraska' to map: Map


Import State census tracts

In [4]:
# 1️⃣ EDIT THESE
item_id = "b0fc7def4bc84d9fac2a0f76cdf1f1ea"   # your AGOL polygon item id
out_name = "NebraskaCensusTracts"               # name inside your project GDB

# 2️⃣ Connect to the same portal that Pro is signed into
gis = GIS("home")

item = gis.content.get(item_id)
if item is None:
    raise ValueError(f"Could not find item with ID {item_id}. "
                     "Check portal sign-in and sharing.")

print(f"Found item: {item.title} ({item.type})")

# 3️⃣ Get the actual feature layer + URL
if hasattr(item, "layers") and len(item.layers) > 0:
    lyr = item.layers[0]   # first sublayer
    print(f"Using sublayer: {lyr.properties.name}")
    in_fc = lyr.url        # <-- use URL for ArcPy
else:
    # item itself might be a Feature Layer
    if hasattr(item, "url"):
        lyr = item
        in_fc = item.url
        print("Item is a Feature Layer; using its URL.")
    else:
        raise TypeError("Item does not have 'layers' or 'url'. "
                        "It may not be a feature layer/service.")

print(f"Feature layer URL for ArcPy: {in_fc}")

# 4️⃣ Figure out output path in the current project GDB
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase     # e.g. C:\...\YourProject.gdb

out_name_valid = arcpy.ValidateTableName(out_name, gdb)
out_fc = f"{gdb}\\{out_name_valid}"
print(f"Output feature class will be: {out_fc}")

# 5️⃣ Run FeatureClassToFeatureClass with error reporting
try:
    arcpy.conversion.FeatureClassToFeatureClass(
        in_features=in_fc,
        out_path=gdb,
        out_name=out_name_valid
    )
    print("FeatureClassToFeatureClass completed successfully.")
except Exception as e:
    print("❌ ArcPy raised an error.")
    print("Python exception:", e)
    print("\nArcPy messages:")
    print(arcpy.GetMessages(2))   # only error + warning messages
    raise

# 6️⃣ Add to active map (optional)
m = aprx.activeMap
if m:
    m.addDataFromPath(out_fc)
    print(f"Added '{out_name_valid}' to map: {m.name}")
else:
    print("No active map found; feature class created but not added to a map.")

Found item: Census Tracts 2020 (Feature Service)
Using sublayer: Tract CENSUS 2020
Feature layer URL for ArcPy: https://gis.ne.gov/Enterprise/rest/services/Census_Tracts_2020/FeatureServer/0
Output feature class will be: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\NebraskaCensusTracts
FeatureClassToFeatureClass completed successfully.
Added 'NebraskaCensusTracts' to map: Map


**Key Layer 1: Wind Power Density**
Import Wind Power Density Dataset using Global Wind Atlas API and Clip it to the Nebraska Polygon

In [5]:

# ---------------------------------------------------
# 1. EDIT THESE
# ---------------------------------------------------

# Global Wind Atlas API URL (GeoTIFF for USA WPD)
# Example placeholder – replace with your actual URL
gwa_url = "https://globalwindatlas.info/api/gis/country/USA/power-density/100"

# Temporary download folder
download_folder = r"C:\Users\shiviana\Documents\ArcGIS\Projects\windfarmsitesuitabilit"
os.makedirs(download_folder, exist_ok=True)
download_tif = os.path.join(download_folder, "gwa_wpd_usa_100m.tif")

# Path to Nebraska feature class *inside your project geodatabase*
# 👉 EDIT THIS to match your actual GDB path
neb_fc = r"C:\Users\shiviana\Documents\ArcGIS\Projects\windfarmsitesuitabilit\windfarmsitesuitabilit.gdb\Nebraska"

# Output raster name (inside your project GDB)
out_raster_name = "WPD_Nebraska_100m"

# ---------------------------------------------------
# 2. Get project + GDB
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

print("Using GDB:", gdb)
print("Nebraska FC:", neb_fc)

# ---------------------------------------------------
# 3. Download the WPD GeoTIFF from GWA API
# ---------------------------------------------------
print("Downloading WPD GeoTIFF from Global Wind Atlas API...")

resp = requests.get(gwa_url, stream=True)
resp.raise_for_status()

with open(download_tif, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print("Downloaded to:", download_tif)

# ---------------------------------------------------
# 4. Load Nebraska FC + WPD raster
# ---------------------------------------------------
wpd_raster = arcpy.Raster(download_tif)
print("WPD raster spatial reference:", wpd_raster.spatialReference.name)
print("Nebraska FC spatial reference:", arcpy.Describe(neb_fc).spatialReference.name)

# ---------------------------------------------------
# 5. Clip using ExtractByMask
# ---------------------------------------------------
arcpy.env.extent = neb_fc
arcpy.env.mask = neb_fc

out_raster_path = os.path.join(gdb, out_raster_name)
print("Clipping WPD to Nebraska →", out_raster_path)

wpd_neb = ExtractByMask(wpd_raster, neb_fc)
wpd_neb.save(out_raster_path)

# Reset env
arcpy.ClearEnvironment("extent")
arcpy.ClearEnvironment("mask")

print("Saved clipped raster to GDB.")

# ---------------------------------------------------
# 6. Add clipped raster to the map
# ---------------------------------------------------
m.addDataFromPath(out_raster_path)
print("Added clipped WPD raster to map.")

Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\windfarmsitesuitabilit\windfarmsitesuitabilit.gdb\Nebraska
Downloaded to: C:\Users\shiviana\Documents\ArcGIS\Projects\windfarmsitesuitabilit\gwa_wpd_usa_100m.tif
WPD raster spatial reference: GCS_WGS_1984
Nebraska FC spatial reference: NAD_1983_StatePlane_Nebraska_FIPS_2600_Feet
Clipping WPD to Nebraska → C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\WPD_Nebraska_100m
Saved clipped raster to GDB.
Added clipped WPD raster to map.


**Key Layer 2: Land Use Land Cover**

In [6]:
# ---------------------------------------------------
# 1. EDIT THESE
# ---------------------------------------------------

# Folder where your LULC TIFF tiles are stored
tiles_folder = r"C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\FinalData\ESA_LULC_Nebraska"

# Path to your saved .lyrx file with the desired LULC color ramp
symbology_layer_file = r"C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\FinalData\symbology\ESA_WorldCover_10m_2021_V200_N39W096_Map.tif.lyrx"

# Path to Nebraska feature class in your geodatabase
neb_fc = r"C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\Nebraska"

# Name for the mosaic raster inside your project GDB
lulc_mosaic_name = "LULC_Mosaic"

# Name for the clipped LULC raster inside your project GDB
lulc_clip_name = "LULC_Nebraska_10m"


# ---------------------------------------------------
# 2. Get project, GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

print("Using GDB:", gdb)
print("Tiles folder:", tiles_folder)
print("Symbology .lyrx:", symbology_layer_file)
print("Nebraska FC:", neb_fc)

# ---------------------------------------------------
# 3. List all TIFF tiles in the folder
# ---------------------------------------------------
arcpy.env.workspace = tiles_folder
tifs = arcpy.ListRasters("*.tif")

if not tifs:
    raise RuntimeError("No .tif rasters found in the specified tiles_folder.")

print("Found TIFF tiles:")
for t in tifs:
    print("  -", t)

tif_paths = [os.path.join(tiles_folder, t) for t in tifs]

# ---------------------------------------------------
# 4. Mosaic TIFFs into a single raster in the GDB
# ---------------------------------------------------
out_mosaic_path = os.path.join(gdb, lulc_mosaic_name)
print("\nCreating mosaic raster in GDB:", out_mosaic_path)

# Use spatial reference from the first tile
first_raster = arcpy.Raster(tif_paths[0])
spatial_ref = first_raster.spatialReference

input_rasters = ";".join(tif_paths)

arcpy.management.MosaicToNewRaster(
    input_rasters=input_rasters,
    output_location=gdb,
    raster_dataset_name_with_extension=lulc_mosaic_name,
    coordinate_system_for_the_raster=spatial_ref,
    pixel_type="8_BIT_UNSIGNED",   # typical for categorical LULC
    number_of_bands=1,
    mosaic_method="LAST",
    mosaic_colormap_mode="FIRST"
)

print("✅ Mosaic LULC raster created in GDB.")

# ---------------------------------------------------
# 5. Add the mosaic to the map & apply symbology
# ---------------------------------------------------
mosaic_layer = None
if m:
    m.addDataFromPath(out_mosaic_path)
    # Find the layer we just added
    for lyr in m.listLayers():
        if lyr.isRasterLayer and lyr.dataSource == out_mosaic_path:
            mosaic_layer = lyr
            break
    print("✅ Added LULC_Mosaic to map.")
else:
    print("ℹ️ No active map; mosaic is saved in GDB only.")

# Apply symbology from .lyrx to mosaic
if mosaic_layer and os.path.exists(symbology_layer_file):
    arcpy.management.ApplySymbologyFromLayer(
        in_layer=mosaic_layer,
        in_symbology_layer=symbology_layer_file
    )
    print("✅ Applied symbology from .lyrx to LULC_Mosaic.")
else:
    if not mosaic_layer:
        print("⚠️ Could not find mosaic layer in the map to apply symbology.")
    if not os.path.exists(symbology_layer_file):
        print("⚠️ .lyrx file not found at the specified path.")


Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Tiles folder: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\FinalData\ESA_LULC_Nebraska
Symbology .lyrx: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\FinalData\symbology\ESA_WorldCover_10m_2021_V200_N39W096_Map.tif.lyrx
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\Nebraska
Found TIFF tiles:
  - ESA_WorldCover_10m_2021_V200_N39W096_Map.tif
  - ESA_WorldCover_10m_2021_V200_N39W099_Map.tif
  - ESA_WorldCover_10m_2021_V200_N39W102_Map.tif
  - ESA_WorldCover_10m_2021_V200_N39W105_Map.tif
  - ESA_WorldCover_10m_2021_V200_N42W099_Map.tif
  - ESA_WorldCover_10m_2021_V200_N42W102_Map.tif
  - ESA_WorldCover_10m_2021_V200_N42W105_Map.tif

Creating mosaic raster in GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\LULC_Mosaic
✅ Mosaic LULC raster created in GDB.
✅ Added LULC_Mosaic to map.
✅ Applied symbology from .lyrx to LULC_Mosaic.


**Key Layer 3: Slope via DEM**

In [7]:


# ---------------------------------------------------
# 1. EDIT THESE
# ---------------------------------------------------

# Path to your final DEM GeoTIFF on disk
in_tif = r"C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\FinalData\Nebraska_USGS3DEP_El_1.tif"

# Name you want for the DEM raster inside the GDB
out_dem_name = "DEM_Nebraska"

# ---------------------------------------------------
# 2. Get project, default GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

out_dem_path = os.path.join(gdb, out_dem_name)

print("Input TIF:", in_tif)
print("Output DEM in GDB:", out_dem_path)

# ---------------------------------------------------
# 3. Copy TIF → GDB raster, preserving scientific values
# ---------------------------------------------------
# Key: 32_BIT_FLOAT and no scaling
arcpy.management.CopyRaster(
    in_raster=in_tif,
    out_rasterdataset=out_dem_path,
    pixel_type="32_BIT_FLOAT",       # preserves elevation as float
    scale_pixel_value="NONE",        # don't stretch/scale values
    colormap_to_RGB="NONE",
    transform="NONE",
    nodata_value=""
)

print("✅ DEM saved to geodatabase:", out_dem_path)

# ---------------------------------------------------
# 4. Add to map
# ---------------------------------------------------
if m:
    m.addDataFromPath(out_dem_path)
    print("✅ Added DEM_Nebraska to map.")
else:
    print("ℹ️ DEM is saved in GDB but no active map was found.")

Input TIF: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\FinalData\Nebraska_USGS3DEP_El_1.tif
Output DEM in GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\DEM_Nebraska
✅ DEM saved to geodatabase: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\DEM_Nebraska
✅ Added DEM_Nebraska to map.


In [8]:
r = arcpy.Raster(r"C:\Users\shiviana\Documents\ArcGIS\Projects\windfarmsitesuitabilit\windfarmsitesuitabilit.gdb\DEM_Nebraska")
print("Real MIN:", arcpy.management.GetRasterProperties(r, "MINIMUM")[0])
print("Real MAX:", arcpy.management.GetRasterProperties(r, "MAXIMUM")[0])

Real MIN: 0
Real MAX: 1677.57153320313


**Key Layer 4: Transmission Lines**

Import Transmission Line Dataset using AGOL item id

In [9]:
#Transmission Lines
# 1️⃣ EDIT THESE
item_id = "d4090758322c4d32a4cd002ffaa0aa12"   # your AGOL polygon item id
out_name = "TransmissionLines"               # name inside your project GDB

# 2️⃣ Connect to the same portal that Pro is signed into
gis = GIS("home")

item = gis.content.get(item_id)
if item is None:
    raise ValueError(f"Could not find item with ID {item_id}. "
                     "Check portal sign-in and sharing.")

print(f"Found item: {item.title} ({item.type})")

# 3️⃣ Get the actual feature layer + URL
if hasattr(item, "layers") and len(item.layers) > 0:
    lyr = item.layers[0]   # first sublayer
    print(f"Using sublayer: {lyr.properties.name}")
    in_fc = lyr.url        # <-- use URL for ArcPy
else:
    # item itself might be a Feature Layer
    if hasattr(item, "url"):
        lyr = item
        in_fc = item.url
        print("Item is a Feature Layer; using its URL.")
    else:
        raise TypeError("Item does not have 'layers' or 'url'. "
                        "It may not be a feature layer/service.")

print(f"Feature layer URL for ArcPy: {in_fc}")

# 4️⃣ Figure out output path in the current project GDB
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase     # e.g. C:\...\YourProject.gdb

out_name_valid = arcpy.ValidateTableName(out_name, gdb)
out_fc = f"{gdb}\\{out_name_valid}"
print(f"Output feature class will be: {out_fc}")

# 5️⃣ Run FeatureClassToFeatureClass with error reporting
try:
    arcpy.conversion.FeatureClassToFeatureClass(
        in_features=in_fc,
        out_path=gdb,
        out_name=out_name_valid
    )
    print("FeatureClassToFeatureClass completed successfully.")
except Exception as e:
    print("❌ ArcPy raised an error.")
    print("Python exception:", e)
    print("\nArcPy messages:")
    print(arcpy.GetMessages(2))   # only error + warning messages
    raise

# 6️⃣ Add to active map (optional)
m = aprx.activeMap
if m:
    m.addDataFromPath(out_fc)
    print(f"Added '{out_name_valid}' to map: {m.name}")
else:
    print("No active map found; feature class created but not added to a map.")

Found item: U.S. Electric Power Transmission Lines (Archive) (Feature Service)
Using sublayer: Electric_Power_Transmission_Lines_A
Feature layer URL for ArcPy: https://services2.arcgis.com/FiaPA4ga0iQKduv3/arcgis/rest/services/US_Electric_Power_Transmission_Lines/FeatureServer/0
Output feature class will be: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionLines
FeatureClassToFeatureClass completed successfully.
Added 'TransmissionLines' to map: Map


clip transmission lines to nebraska

In [10]:
# clip transmission lines to nebraska

# ---------------------------------------------------
# 1. EDIT THIS: path to Nebraska feature class
# ---------------------------------------------------
neb_fc = r"C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\Nebraska"

# ---------------------------------------------------
# 2. Get project, default GDB, and map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

print("Using GDB:", gdb)
print("Nebraska FC:", neb_fc)

# Input TransmissionLines feature class in the GDB
in_trans = os.path.join(gdb, "TransmissionLines")

# Output clipped feature class
out_trans_name = "TransmissionLines_Nebraska"
out_trans = os.path.join(gdb, out_trans_name)

# ---------------------------------------------------
# 3. Clip TransmissionLines to Nebraska
# ---------------------------------------------------
print(f"Clipping {in_trans} to Nebraska → {out_trans}")

arcpy.analysis.Clip(
    in_features=in_trans,
    clip_features=neb_fc,
    out_feature_class=out_trans
)

print("✅ Saved clipped transmission lines to GDB:", out_trans)

# ---------------------------------------------------
# 4. Add result to the map
# ---------------------------------------------------
if m:
    m.addDataFromPath(out_trans)
    print("✅ Added clipped TransmissionLines layer to map:", m.name)
else:
    print("ℹ️ No active map; output saved in GDB only.")

Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject\MyProject.gdb\Nebraska
Clipping C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionLines to Nebraska → C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionLines_Nebraska
✅ Saved clipped transmission lines to GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionLines_Nebraska
✅ Added clipped TransmissionLines layer to map: Map


select only transmission lines that are greater than or equal to 115 kV and clean data

In [11]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

# Input FC in your GDB
input_fc = os.path.join(gdb, "TransmissionLines_Nebraska")

for f in arcpy.ListFields(input_fc):
    print(f.name)

OBJECTID_1
Shape
OBJECTID
ID
TYPE
STATUS
NAICS_CODE
NAICS_DESC
SOURCE
SOURCEDATE
VAL_METHOD
VAL_DATE
OWNER
VOLTAGE
VOLT_CLASS
INFERRED
SUB_1
SUB_2
Shape__Len
Shape_Length


In [12]:
# ---------------------------------------------------
# 1. Setup project + paths
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

# Input FC in your GDB
input_fc = os.path.join(gdb, "TransmissionLines_Nebraska")

# Output FC
output_fc = os.path.join(gdb, "TransmissionNebraskaCleaned")

print("Using GDB:", gdb)
print("Input:", input_fc)
print("Output:", output_fc)

# ---------------------------------------------------
# 2. Build SQL query
# ---------------------------------------------------
# Keep only records where:
# Voltage(Kilovolts) != -999999 AND Voltage(Kilovolts) >= 115

sql_query = 'VOLTAGE <> -999999 AND VOLTAGE >= 115'
print("SQL query:", sql_query)

# ---------------------------------------------------
# 3. Run Select (creates clean copy)
# ---------------------------------------------------
arcpy.analysis.Select(
    in_features=input_fc,
    out_feature_class=output_fc,
    where_clause=sql_query
)

print("✅ Cleaned transmission line feature class created.")

# ---------------------------------------------------
# 4. Add output to map
# ---------------------------------------------------
if m:
    m.addDataFromPath(output_fc)
    print("✅ Added cleaned layer to map:", output_fc)
else:
    print("ℹ️ Output saved in GDB but no active map found.")


Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Input: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionLines_Nebraska
Output: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionNebraskaCleaned
SQL query: VOLTAGE <> -999999 AND VOLTAGE >= 115
✅ Cleaned transmission line feature class created.
✅ Added cleaned layer to map: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionNebraskaCleaned


**Reprojecting all Preprocessed Base Layers**

In [13]:

# ------------------------------------------------------------------
# 0. PROJECT / GDB CONTEXT
# ------------------------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

print("Working GDB:", gdb)

# Nebraska working projection: NAD 1983 / UTM Zone 14N (EPSG: 26914)
nebraska_sr = arcpy.SpatialReference(26914)


# ------------------------------------------------------------------
# 1. DEFINE INPUT NAMES (AS THEY EXIST IN YOUR GDB)
#    Adjust names here if yours differ.
# ------------------------------------------------------------------
dem_in  = os.path.join(gdb, "DEM_Nebraska")
lulc_in = os.path.join(gdb, "LULC_Mosaic")
wpd_in  = os.path.join(gdb, "WPD_Nebraska_100m")   # ← change if your WPD raster has a different name
trans_in = os.path.join(gdb, "TransmissionNebraskaCleaned")

dem_out  = os.path.join(gdb, "DEM_Nebraska_UTM14")
lulc_out = os.path.join(gdb, "LULC_Mosaic_UTM14")
wpd_out  = os.path.join(gdb, "WPD_Nebraska_100m_UTM14")
trans_out = os.path.join(gdb, "TransmissionLines_UTM14")


# ------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# ------------------------------------------------------------------
def project_raster(in_raster, out_raster, resampling_type="NEAREST", cell_size=None):
    if not arcpy.Exists(in_raster):
        print(f"⚠️ Raster not found, skipping: {in_raster}")
        return
    print(f"Projecting raster: {os.path.basename(in_raster)} → {os.path.basename(out_raster)}")
    arcpy.management.ProjectRaster(
        in_raster=in_raster,
        out_raster=out_raster,
        out_coor_system=nebraska_sr,
        resampling_type=resampling_type,
        cell_size=cell_size
    )

def project_vector(in_fc, out_fc):
    if not arcpy.Exists(in_fc):
        print(f"⚠️ Feature class not found, skipping: {in_fc}")
        return
    print(f"Projecting vector: {os.path.basename(in_fc)} → {os.path.basename(out_fc)}")
    arcpy.management.Project(
        in_dataset=in_fc,
        out_dataset=out_fc,
        out_coor_system=nebraska_sr
    )


# ------------------------------------------------------------------
# 3. RUN REPROJECTIONS
# ------------------------------------------------------------------

# DEM – continuous → BILINEAR
project_raster(dem_in, dem_out, resampling_type="BILINEAR")

# LULC – categorical → NEAREST
project_raster(lulc_in, lulc_out, resampling_type="NEAREST")

# WPD – continuous → BILINEAR
project_raster(wpd_in, wpd_out, resampling_type="BILINEAR")

# Transmission lines – vector
project_vector(trans_in, trans_out)

print("✅ Reprojection complete. New projected layers saved in the same GDB.")


Working GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Projecting raster: DEM_Nebraska → DEM_Nebraska_UTM14
Projecting raster: LULC_Mosaic → LULC_Mosaic_UTM14
Projecting raster: WPD_Nebraska_100m → WPD_Nebraska_100m_UTM14
Projecting vector: TransmissionNebraskaCleaned → TransmissionLines_UTM14
✅ Reprojection complete. New projected layers saved in the same GDB.


**Reclassifying the Data Layers**

**Slope**

In [17]:
import os
import arcpy
from arcpy.sa import Slope, Reclassify, RemapRange

# ---------------------------------------------------
# 1. Get project, GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

# Paths to inputs in your GDB
dem_path   = os.path.join(gdb, "DEM_Nebraska_UTM14")   # DEM raster
neb_fc     = os.path.join(gdb, "Nebraska")       # Nebraska polygon

# Output names
slope_name      = "Slope_deg"
slope_suit_name = "Slope_Suitability"

slope_path      = os.path.join(gdb, slope_name)
slope_suit_path = os.path.join(gdb, slope_suit_name)

print("Using GDB:", gdb)
print("DEM:", dem_path)
print("Nebraska FC:", neb_fc)

# ---------------------------------------------------
# 2. Set environments (clip processing to Nebraska)
# ---------------------------------------------------
arcpy.CheckOutExtension("Spatial")

arcpy.env.extent = neb_fc
arcpy.env.mask   = neb_fc

# ---------------------------------------------------
# 3. Create Slope raster (degrees)
# ---------------------------------------------------
print("\nCreating slope raster in degrees...")

slope_raster = Slope(
    in_raster=dem_path,
    output_measurement="DEGREE",   # slope in degrees
    z_factor=1                     # DEM is already in consistent vertical units
)

slope_raster.save(slope_path)
print("✅ Saved slope raster to:", slope_path)

# ---------------------------------------------------
# 4. Reclassify Slope into suitability scores (1–5)
# ---------------------------------------------------
print("\nReclassifying slope into suitability scores...")

# Classes (degrees):
# 0–3   → 5 (Excellent)
# 3–6   → 4 (Good)
# 6–9   → 3 (Moderate)
# 9–12  → 2 (Poor)
# 12–90 → 1 (Unsuitable)

remap = RemapRange([
    [0,   3,   5],   # flat
    [3,   6,   4],
    [6,   9,   3],
    [9,   12,  2],
    [12,  90,  1]   # assume max slope < 90°
])

slope_suit = Reclassify(
    in_raster=slope_raster,
    reclass_field="VALUE",
    remap=remap,
    missing_values="NODATA"
)

slope_suit.save(slope_suit_path)
print("✅ Saved slope suitability raster to:", slope_suit_path)

# ---------------------------------------------------
# 5. Clear environments
# ---------------------------------------------------
arcpy.ClearEnvironment("extent")
arcpy.ClearEnvironment("mask")

# ---------------------------------------------------
# 6. Add outputs to the map
# ---------------------------------------------------
if m:
    m.addDataFromPath(slope_path)
    m.addDataFromPath(slope_suit_path)
    print("✅ Added Slope_deg and Slope_Suitability to map.")
else:
    print("ℹ️ Outputs saved in GDB; no active map found.")

Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
DEM: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\DEM_Nebraska_UTM14
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Nebraska

Creating slope raster in degrees...
✅ Saved slope raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Slope_deg

Reclassifying slope into suitability scores...
✅ Saved slope suitability raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Slope_Suitability
✅ Added Slope_deg and Slope_Suitability to map.


**Transmission Lines Distance Vector + Reclassification**

In [16]:
import os
import arcpy
from arcpy.sa import EucDistance, Reclassify, RemapRange

# ---------------------------------------------------
# 1. Get project, GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

# Paths in your GDB
trans_fc   = os.path.join(gdb, "TransmissionLines_UTM14")  # cleaned ≥115 kV lines
neb_fc     = os.path.join(gdb, "Nebraska")                     # AOI polygon

# Output names
dist_name       = "Trans_Distance_m"
dist_suit_name  = "Trans_Distance_Suitability"

dist_path       = os.path.join(gdb, dist_name)
dist_suit_path  = os.path.join(gdb, dist_suit_name)

print("Using GDB:", gdb)
print("Transmission FC:", trans_fc)
print("Nebraska FC:", neb_fc)

# ---------------------------------------------------
# 2. (Optional but good) Check spatial units
# ---------------------------------------------------
sr = arcpy.Describe(trans_fc).spatialReference
print(f"Transmission spatial reference: {sr.name}, linear unit: {sr.linearUnitName}")

# We assume the data is in a projected CRS with meters as linear unit.
# If it's in degrees (e.g., WGS 1984), you should project it first.

# ---------------------------------------------------
# 3. Set environments (extent & mask to Nebraska)
# ---------------------------------------------------
arcpy.CheckOutExtension("Spatial")

arcpy.env.extent = neb_fc
arcpy.env.mask   = neb_fc

# ---------------------------------------------------
# 4. Create Euclidean distance raster (meters)
# ---------------------------------------------------
print("\nCreating Euclidean distance raster from transmission lines...")

# EucDistance(in_source_data, {maximum_distance}, {cell_size}, {out_direction_raster})
dist_raster = EucDistance(
    in_source_data=trans_fc,
    maximum_distance="",    # let it compute up to full needed distance
    cell_size=""            # default (will match env / source)
)

dist_raster.save(dist_path)
print("✅ Saved distance raster to:", dist_path)

# ---------------------------------------------------
# 5. Reclassify distance to suitability scores
# ---------------------------------------------------
print("\nReclassifying distance raster into suitability scores...")

# Distances in **meters**:
# 0–3000 m   → 5  (0–3 km)
# 3000–6000  → 4  (3–6 km)
# 6000–10000 → 3  (6–10 km)
# 10000–20000→ 2  (10–20 km)
# >20000     → 1  ( >20 km )

remap = RemapRange([
    [0,     3000,   5],
    [3000,  6000,   4],
    [6000,  10000,  3],
    [10000, 20000,  2],
    [20000, 999999, 1]   # upper bound high enough to cover all distances
])

dist_suit = Reclassify(
    in_raster=dist_raster,
    reclass_field="VALUE",
    remap=remap,
    missing_values="NODATA"
)

dist_suit.save(dist_suit_path)
print("✅ Saved transmission distance suitability raster to:", dist_suit_path)

# ---------------------------------------------------
# 6. Clear environments
# ---------------------------------------------------
arcpy.ClearEnvironment("extent")
arcpy.ClearEnvironment("mask")

# ---------------------------------------------------
# 7. Add outputs to the map
# ---------------------------------------------------
if m:
    m.addDataFromPath(dist_path)
    m.addDataFromPath(dist_suit_path)
    print("✅ Added Trans_Distance_m and Trans_Distance_Suitability to map.")
else:
    print("ℹ️ Outputs saved in GDB; no active map found.")

Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
Transmission FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\TransmissionLines_UTM14
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Nebraska
Transmission spatial reference: NAD_1983_UTM_Zone_14N, linear unit: Meter

Creating Euclidean distance raster from transmission lines...
✅ Saved distance raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Trans_Distance_m

Reclassifying distance raster into suitability scores...
✅ Saved transmission distance suitability raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Trans_Distance_Suitability
✅ Added Trans_Distance_m and Trans_Distance_Suitability to map.


**WPD Reclassify**

In [19]:

# ---------------------------------------------------
# 1. Get project, GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

# Paths in your GDB
wpd_path      = os.path.join(gdb, "WPD_Nebraska_100m_UTM14")  # your clipped WPD raster
neb_fc        = os.path.join(gdb, "Nebraska")

# Output name
wpd_suit_name = "WPD_Suitability"
wpd_suit_path = os.path.join(gdb, wpd_suit_name)

print("Using GDB:", gdb)
print("WPD raster:", wpd_path)
print("Nebraska FC:", neb_fc)

# ---------------------------------------------------
# 2. Set environments (clip + mask to Nebraska)
# ---------------------------------------------------
arcpy.CheckOutExtension("Spatial")

arcpy.env.extent = neb_fc
arcpy.env.mask   = neb_fc

# ---------------------------------------------------
# 3. Reclassify WPD into suitability scores (1–5)
# ---------------------------------------------------
print("\nReclassifying WPD into suitability scores...")

# Classes (W/m²):
# < 200      → 1
# 200–299    → 2
# 300–449    → 3
# 450–599    → 4
# ≥ 600      → 5

remap = RemapRange([
    [0,    200,  1],   # note: if there are true zeros, they go here
    [200,  300,  2],
    [300,  450,  3],
    [450,  600,  4],
    [600,  99999, 5]   # upper bound high enough for all realistic WPD values
])

wpd_raster = arcpy.Raster(wpd_path)

wpd_suit_raw = Reclassify(
    in_raster=wpd_raster,
    reclass_field="VALUE",
    remap=remap,
    missing_values="NODATA"   # leave NoData as NoData for now
)

# ---------------------------------------------------
# 4. Convert NoData to 0 (explicitly mark unusable / outside area)
# ---------------------------------------------------
print("Assigning 0 to NoData areas...")

wpd_suit_final = Con(
    IsNull(wpd_raster),   # where original WPD is NoData
    0,                    # → set suitability = 0
    wpd_suit_raw          # else keep classified value (1–5)
)

wpd_suit_final.save(wpd_suit_path)
print("✅ Saved WPD suitability raster to:", wpd_suit_path)

# ---------------------------------------------------
# 5. Clear environments
# ---------------------------------------------------
arcpy.ClearEnvironment("extent")
arcpy.ClearEnvironment("mask")

# ---------------------------------------------------
# 6. Add output to map
# ---------------------------------------------------
if m:
    m.addDataFromPath(wpd_suit_path)
    print("✅ Added WPD_Suitability to map.")
else:
    print("ℹ️ Output saved in GDB; no active map found.")


Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
WPD raster: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\WPD_Nebraska_100m_UTM14
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Nebraska

Reclassifying WPD into suitability scores...
Assigning 0 to NoData areas...
✅ Saved WPD suitability raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\WPD_Suitability
✅ Added WPD_Suitability to map.


**LULC Reclassify**

In [20]:

# ---------------------------------------------------
# 1. Get project, GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

# 🔹 Paths in your GDB
lulc_path      = os.path.join(gdb, "LULC_Mosaic_UTM14")   # your clipped Sentinel LULC
neb_fc         = os.path.join(gdb, "Nebraska")            # AOI polygon

# 🔹 Output
lulc_suit_name = "LULC_Suitability"
lulc_suit_path = os.path.join(gdb, lulc_suit_name)

print("Using GDB:", gdb)
print("LULC raster:", lulc_path)
print("Nebraska FC:", neb_fc)

# ---------------------------------------------------
# 2. EDIT THESE: Sentinel LULC numeric codes
# ---------------------------------------------------
# 👉 Replace these placeholder codes with the actual ones from your raster
#    (Layer > Symbology > Unique Values to see them)

CODE_WATER             = 80
CODE_TREES             = 10
CODE_FLOODED_VEG       = 60
CODE_CROPS             = 40
CODE_BUILT_AREA        = 50
CODE_BARE_GROUND       = 83
CODE_SNOW_ICE          = 84
CODE_CLOUDS            = 85
CODE_RANGELAND         = 30

# ---------------------------------------------------
# 3. Build suitability mapping (Sentinel → Suitability)
# ---------------------------------------------------
# Suitability:
# Crops, Rangeland      → 5
# Bare Ground           → 3
# Trees                 → 2
# Flooded Vegetation    → 1
# Water, Built, Snow/Ice,
# Clouds (artifact)     → 0

remap = RemapValue([
    [CODE_CROPS,          5],
    [CODE_RANGELAND,      5],
    [CODE_BARE_GROUND,    3],
    [CODE_TREES,          2],
    [CODE_FLOODED_VEG,    1],
    [CODE_WATER,          0],
    [CODE_BUILT_AREA,     0],
    [CODE_SNOW_ICE,       0],
    [CODE_CLOUDS,         0]
])

# ---------------------------------------------------
# 4. Set environments & run Reclassify
# ---------------------------------------------------
arcpy.CheckOutExtension("Spatial")

arcpy.env.extent = neb_fc
arcpy.env.mask   = neb_fc

print("\nReclassifying LULC into suitability scores...")

lulc_raster = arcpy.Raster(lulc_path)

lulc_suit = Reclassify(
    in_raster=lulc_raster,
    reclass_field="VALUE",   # LULC code is in VALUE
    remap=remap,
    missing_values="NODATA"  # any unexpected codes become NoData
)

lulc_suit.save(lulc_suit_path)
print("✅ Saved LULC suitability raster to:", lulc_suit_path)

# ---------------------------------------------------
# 5. Clear env & add to map
# ---------------------------------------------------
arcpy.ClearEnvironment("extent")
arcpy.ClearEnvironment("mask")

if m:
    m.addDataFromPath(lulc_suit_path)
    print("✅ Added LULC_Suitability to map.")
else:
    print("ℹ️ Output saved in GDB; no active map found.")

Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
LULC raster: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\LULC_Mosaic_UTM14
Nebraska FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Nebraska

Reclassifying LULC into suitability scores...
✅ Saved LULC suitability raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\LULC_Suitability
✅ Added LULC_Suitability to map.


**Weighted Overlay**

In [21]:

# ---------------------------------------------------
# 1. Get project, GDB, map
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
m = aprx.activeMap

neb_fc = os.path.join(gdb, "Nebraska")

# Paths to suitability rasters in GDB
wpd_suit_path   = os.path.join(gdb, "WPD_Suitability")
slope_suit_path = os.path.join(gdb, "Slope_Suitability")
trans_suit_path = os.path.join(gdb, "Trans_Distance_Suitability")
lulc_suit_path  = os.path.join(gdb, "LULC_Suitability")

# Output
final_name      = "Suitability_Final"
final_path      = os.path.join(gdb, final_name)

print("Using GDB:", gdb)
print("WPD:",   wpd_suit_path)
print("Slope:", slope_suit_path)
print("Trans:", trans_suit_path)
print("LULC:",  lulc_suit_path)

# ---------------------------------------------------
# 2. Set environments (clip + mask to Nebraska)
# ---------------------------------------------------
arcpy.CheckOutExtension("Spatial")
arcpy.env.extent = neb_fc
arcpy.env.mask   = neb_fc

# ---------------------------------------------------
# 3. Build weighted sum
# ---------------------------------------------------
print("\nComputing weighted suitability raster...")

wpd   = Raster(wpd_suit_path)
slope = Raster(slope_suit_path)
trans = Raster(trans_suit_path)
lulc  = Raster(lulc_suit_path)

# Weights:
# WPD:   0.4
# Trans: 0.3
# Slope: 0.2
# LULC:  0.1

final_suit = (wpd * 0.4) + (trans * 0.3) + (slope * 0.1) + (lulc * 0.1)

final_suit.save(final_path)
print("✅ Saved final suitability raster to:", final_path)

# ---------------------------------------------------
# 4. (Optional) round to nearest integer 1–5
# ---------------------------------------------------
final_int_name = "Suitability_Final_Int"
final_int_path = os.path.join(gdb, final_int_name)

from arcpy.sa import Int

final_suit_int = Int(final_suit + 0.5)   # simple rounding
final_suit_int.save(final_int_path)
print("✅ Saved integer suitability raster to:", final_int_path)

# ---------------------------------------------------
# 5. Clear env & add to map
# ---------------------------------------------------
arcpy.ClearEnvironment("extent")
arcpy.ClearEnvironment("mask")

if m:
    m.addDataFromPath(final_path)
    m.addDataFromPath(final_int_path)
    print("✅ Added Suitability_Final and Suitability_Final_Int to map.")
else:
    print("ℹ️ Outputs saved in GDB; no active map found.")

Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb
WPD: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\WPD_Suitability
Slope: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Slope_Suitability
Trans: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Trans_Distance_Suitability
LULC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\LULC_Suitability

Computing weighted suitability raster...
✅ Saved final suitability raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Suitability_Final
✅ Saved integer suitability raster to: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Suitability_Final_Int
✅ Added Suitability_Final and Suitability_Final_Int to map.


**Post Analysis**

**Extract the top suitability zones (classes 4 and 5)**

In [25]:

# values ≥4:

final = Raster(os.path.join(gdb, "Suitability_Final_Int"))
top_zones = Con(final >= 4, 1)

top_zones.save(os.path.join(gdb, "HighSuitabilityAreas"))


**Convert high-suitability (classes 4 and 5) areas to polygons**

In [26]:


arcpy.RasterToPolygon_conversion(
    in_raster=os.path.join(gdb, "HighSuitabilityAreas"),
    out_polygon_features=os.path.join(gdb, "HighSuitabilityAreas_Poly"),
    simplify="NO_SIMPLIFY"
)


<Result 'C:\\Users\\shiviana\\Documents\\ArcGIS\\Projects\\MyProject1\\MyProject1.gdb\\HighSuitabilityAreas_Poly'>

**Extract the topmost suitability zone (class 5)**

In [27]:

# values >4:

final = Raster(os.path.join(gdb, "Suitability_Final_Int"))
topmost_zones = Con(final > 4, 1)

topmost_zones.save(os.path.join(gdb, "HighestSuitabilityAreas"))


**Convert highest-suitability (class 5) areas to polygons**

In [28]:


arcpy.RasterToPolygon_conversion(
    in_raster=os.path.join(gdb, "HighestSuitabilityAreas"),
    out_polygon_features=os.path.join(gdb, "HighestSuitabilityAreas_Poly"),
    simplify="NO_SIMPLIFY"
)


<Result 'C:\\Users\\shiviana\\Documents\\ArcGIS\\Projects\\MyProject1\\MyProject1.gdb\\HighestSuitabilityAreas_Poly'>

**Compute summary statistics**

In [35]:
zone_fc = os.path.join(gdb, "Nebraska")
for f in arcpy.ListFields(zone_fc):
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
ID1 Integer
GlobalID Guid
STArea__ Double
STLength__ Double
Shape_Length Double
Shape_Area Double


In [37]:
# Paths
zone_fc      = os.path.join(gdb, "Nebraska")              # or "Nebraska_Counties" if that's your zones
zone_field   = "OBJECTID"                                     # change if your ID field is different
class_raster = os.path.join(gdb, "Suitability_Final_Int") # your integer suitability raster
class_field  = "Value"                                    # field in the raster with class values

out_tab = os.path.join(gdb, "Suitability_Area_Table")

print("Zone FC:", zone_fc)
print("Class raster:", class_raster)
print("Output table:", out_tab)

# Run Tabulate Area with *positional* arguments in correct order
arcpy.gp.TabulateArea_sa(
    zone_fc,        # in_zone_data
    zone_field,     # zone_field
    class_raster,   # in_class_data
    class_field,    # class_field
    out_tab         # out_table
    # optional: processing cell size (we can omit and let ArcGIS infer)
)

print("✅ Created tabulated area table:", out_tab)

Zone FC: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Nebraska
Class raster: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Suitability_Final_Int
Output table: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Suitability_Area_Table
✅ Created tabulated area table: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Suitability_Area_Table


**Printing Summary Statistics: Quantifying Wind Turbine Site Suitability Area in the State of Nebraska**

In [51]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase

tab = os.path.join(gdb, "Suitability_Area_Table")

print("Reading:", tab)

# --- Collect area fields (Tabulate Area output fields begin with VALUE_) ---
fields = [f.name for f in arcpy.ListFields(tab) if f.name.startswith("VALUE_")]

# --- Read the single row of the table (Nebraska) ---
with arcpy.da.SearchCursor(tab, fields) as cursor:
    row = next(cursor)  # only one row
    areas = dict(zip(fields, row))

# --- Compute totals + percentages ---
total_area = sum(areas.values())

print("\n=== Suitability Area Percentages (0–5) ===\n")
for f_name, area in areas.items():
    class_val = f_name.replace("VALUE_", "")  # e.g., VALUE_3 → 3
    pct = (area / total_area) * 100 if total_area > 0 else 0
    print(f"Class {class_val}: {pct:.2f}% of Nebraska")

print("\nTotal mapped area:", f"{total_area:,.0f}", "square units (raster cell area summary)")

Reading: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb\Suitability_Area_Table

=== Suitability Area Percentages (0–5) ===

Class 1: 0.01% of Nebraska
Class 2: 2.29% of Nebraska
Class 3: 59.89% of Nebraska
Class 4: 37.33% of Nebraska
Class 5: 0.49% of Nebraska

Total mapped area: 200,258,598,690 square units (raster cell area summary)


In [53]:
# Exporting Summary Statistics Table to AGOL as a hosted feature table

table_path = os.path.join(gdb, "Suitability_Area_Table")

# Export GDB table → .csv
csv_path = os.path.join(os.path.dirname(gdb), "Suitability_Area_Table.csv")
arcpy.conversion.TableToTable(table_path, os.path.dirname(gdb), "Suitability_Area_Table.csv")

# Upload CSV to AGOL as a hosted table
csv_item = gis.content.add(
    item_properties={
        "title": "Suitability Area Table",
        "type": "CSV",
        "tags": "Nebraska, Suitability, Statistics",
        "snippet": "Summary statistics for suitability classes"
    },
    data=csv_path
)

# Publish as hosted feature table (this creates a Feature Layer with no geometry)
table_layer = csv_item.publish()

print("Uploaded + published:", table_layer.url)

C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\IPython\core\interactiveshell.py:3550: DeprecatedWarning: add is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.add()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Uploaded + published: https://services1.arcgis.com/ZIL9uO234SBBPGL7/arcgis/rest/services/Suitability_Area_Table/FeatureServer


**Save final layers to AGOL**

In [50]:
gis = GIS("home")
me = gis.users.me

# ---------------------------------------------------
# 0. PROJECT / GDB CONTEXT
# ---------------------------------------------------
aprx = arcpy.mp.ArcGISProject("CURRENT")
gdb = aprx.defaultGeodatabase
arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

print("Using GDB:", gdb)

# Create a temporary map to publish one layer at a time
def make_temp_map():
    # Create a new TOC map, blank
    temp_map = aprx.createMap("TEMP_PUBLISH_MAP")
    return temp_map

# ---------------------------------------------------
# 1. LAYER DEFINITIONS
# ---------------------------------------------------
layers_to_publish = [
    # name_in_gdb                  service_title                     service_type
    ("Suitability_Final_Int",      "Suitability Final Nebraska",   "TILE"),
    ("HighSuitabilityAreas_Poly",  "High Suitability Areas",         "FEATURE"),
    ("HighestSuitabilityAreas_Poly","Highest Suitability Areas",     "FEATURE"),
    ("Nebraska",                   "Nebraska Boundary",              "FEATURE"),
    ("NebraskaCensusTracts",       "Nebraska Census Tracts",         "FEATURE"),
    ("LULC_Suitability",           "Land Use Land Cover LULC Suitability",               "TILE"),
    ("WPD_Suitability",            "Wind Power Density WPD Suitability",                "TILE"),
    ("Slope_Suitability",          "Slope Suitability",              "TILE"),
    ("Trans_Distance_Suitability", "Distance from High Voltage Transmission Line Suitability", "TILE"),
    ("DEM_Nebraska_UTM14",         "Digital Elevation Model DEM Nebraska Raw Elevation Values meters",             "TILE"),
    ("LULC_Mosaic_UTM14",          "Land Use Land Cover LULC Mosaic Raw",              "TILE"),
    ("TransmissionLines_UTM14",    "Transmission Lines 115kV or greater",    "FEATURE"),
    ("WPD_Nebraska_100m_UTM14",    "Wind Power Density WPD Nebraska at 100m height Raw Watts per m2",        "TILE")
]

sd_folder = os.path.join(os.path.dirname(gdb), "SD")
os.makedirs(sd_folder, exist_ok=True)

# ---------------------------------------------------
# 2. PUBLISH FUNCTION
# ---------------------------------------------------
def publish_web_layer(dataset_name, service_title, service_type,
                      summary="", tags="Nebraska, Wind, Suitability"):

    in_path = os.path.join(gdb, dataset_name)
    if not arcpy.Exists(in_path):
        print(f"⚠️ Skipping '{dataset_name}' – not found.")
        return None

    # --- NEW: check if a hosted item with this title already exists ---
    existing = gis.content.search(
        query=f'title:"{service_title}" AND owner:{me.username}',
        max_items=1
    )
    if existing:
        print(f"\n⏭ Skipping '{service_title}' – already exists in AGOL.")
        # Still return the title so it can be added to the web map later
        return service_title

    print(f"\n🔹 Publishing '{dataset_name}' → {service_type} service.")

    # Create new temporary map with only this layer
    temp_map = make_temp_map()
    lyr = temp_map.addDataFromPath(in_path)

    # Create service definition draft
    sddraft = temp_map.getWebLayerSharingDraft(
        "HOSTING_SERVER",
        service_type,
        service_title,
        [lyr]
    )

    sddraft.summary = summary or service_title
    sddraft.tags = tags
    sddraft.copyDataToServer = True

    safe_name = service_title.replace(" ", "_").replace("(", "").replace(")", "")
    sddraft_path = os.path.join(sd_folder, f"{safe_name}.sddraft")
    sd_path = os.path.join(sd_folder, f"{safe_name}.sd")

    sddraft.exportToSDDraft(sddraft_path)
    arcpy.StageService_server(sddraft_path, sd_path)
    arcpy.UploadServiceDefinition_server(sd_path, "My Hosted Services")

    print(f"  ✅ Published: {service_title}")
    return service_title

# ---------------------------------------------------
# 3. PUBLISH ALL LAYERS
# ---------------------------------------------------
published_titles = []
for dataset_name, title, s_type in layers_to_publish:
    t = publish_web_layer(dataset_name, title, s_type)
    if t:
        published_titles.append(t)

print("\n✔ All layers processed.")


Using GDB: C:\Users\shiviana\Documents\ArcGIS\Projects\MyProject1\MyProject1.gdb

⏭ Skipping 'Suitability Final Nebraska' – already exists in AGOL.

⏭ Skipping 'High Suitability Areas' – already exists in AGOL.

⏭ Skipping 'Highest Suitability Areas' – already exists in AGOL.

⏭ Skipping 'Nebraska Boundary' – already exists in AGOL.

⏭ Skipping 'Nebraska Census Tracts' – already exists in AGOL.

⏭ Skipping 'Land Use Land Cover LULC Suitability' – already exists in AGOL.

⏭ Skipping 'Wind Power Density WPD Suitability' – already exists in AGOL.

⏭ Skipping 'Slope Suitability' – already exists in AGOL.

⏭ Skipping 'Distance from High Voltage Transmission Line Suitability' – already exists in AGOL.

🔹 Publishing 'DEM_Nebraska_UTM14' → TILE service.
  ✅ Published: Digital Elevation Model DEM Nebraska Raw Elevation Values meters

🔹 Publishing 'LULC_Mosaic_UTM14' → TILE service.
  ✅ Published: Land Use Land Cover LULC Mosaic Raw

🔹 Publishing 'TransmissionLines_UTM14' → FEATURE service.
  ✅ 